# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name:
Date:

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [1]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

# Project root is one level above the notebooks folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

CHECKS = [
    (".env", "NEEDED", "Local environment variables file"),
    (".env.example", "NEEDED", "Template for environment variables"),
]

print(f"Project root: {ROOT}\n")

missing = 0

for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()

    if not here and kind == "NEEDED":
        missing += 1

    print(
        f"[{'OK' if here else 'MISS'}] "
        f"{kind:<8} {rel:<20} {note}"
    )

if missing:
    print(f"\n{missing} needed file(s) missing.")
else:
    print("\nAll needed files present.")

Project root: /Users/crablan/Desktop/bootcamp_haoting_lan

[OK] NEEDED   .env                 Local environment variables file
[OK] NEEDED   .env.example         Template for environment variables

All needed files present.


In [2]:
import os
import pathlib
import datetime as dt
import requests
import pandas as pd

from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

load_dotenv(ROOT / ".env")

print(
    "ALPHAVANTAGE_API_KEY loaded?",
    bool(os.getenv("ALPHAVANTAGE_API_KEY"))
)

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [7]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [8]:
SYMBOL = "AAPL"
USE_ALPHA = bool(os.getenv("ALPHAVANTAGE_API_KEY"))

try:
    if USE_ALPHA:
        url = "https://www.alphavantage.co/query"

        params = {
            "function": "TIME_SERIES_DAILY_ADJUSTED",
            "symbol": SYMBOL,
            "outputsize": "compact",
            "apikey": os.getenv("ALPHAVANTAGE_API_KEY")
        }

        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        js = r.json()

        key = [k for k in js if "Time Series" in k][0]

        df_api = (
            pd.DataFrame(js[key])
            .T
            .reset_index()
            .rename(
                columns={
                    "index": "date",
                    "5. adjusted close": "adj_close"
                }
            )[["date", "adj_close"]]
        )

    else:
        import yfinance as yf

        df_api = yf.download(
            SYMBOL,
            period="3mo",
            interval="1d",
            auto_adjust=False,
            progress=False,
            multi_level_index=False
        ).reset_index()

        df_api = df_api[["Date", "Adj Close"]]
        df_api.columns = ["date", "adj_close"]

    # Parse data types
    df_api["date"] = pd.to_datetime(df_api["date"])
    df_api["adj_close"] = pd.to_numeric(
        df_api["adj_close"],
        errors="coerce"
    )

    print(df_api.head())
    print("\nData types:")
    print(df_api.dtypes)

except Exception as e:
    print("API ingestion failed:", e)
required_api_cols = ["date", "adj_close"]

v_api = validate(df_api, required_api_cols)

print("\nValidation results:")
print(v_api)

print("\nNA counts:")
print(df_api[required_api_cols].isna().sum())

print("\nBasic rules:")
print("Rows greater than 0:", len(df_api) > 0)
print("Dates valid:", df_api["date"].notna().all())
print("Prices positive:", (df_api["adj_close"].dropna() > 0).all())

        date   adj_close
0 2026-05-18  297.583344
1 2026-05-19  298.712372
2 2026-05-20  301.989563
3 2026-05-21  304.727173
4 2026-05-22  308.553894

Data types:
date         datetime64[s]
adj_close          float64
dtype: object

Validation results:
{'missing': [], 'shape': (63, 2), 'na_total': 0}

NA counts:
date         0
adj_close    0
dtype: int64

Basic rules:
Rows greater than 0: True
Dates valid: True
Prices positive: True


In [9]:
required_api_cols = ["date", "adj_close"]

v_api = validate(df_api, required_api_cols)

print("\nValidation results:")
print(v_api)

print("\nNA counts:")
print(df_api[required_api_cols].isna().sum())

print("\nBasic rules:")
print("Rows greater than 0:", len(df_api) > 0)
print("Dates valid:", df_api["date"].notna().all())
print("Prices positive:", (df_api["adj_close"].dropna() > 0).all())


Validation results:
{'missing': [], 'shape': (63, 2), 'na_total': 0}

NA counts:
date         0
adj_close    0
dtype: int64

Basic rules:
Rows greater than 0: True
Dates valid: True
Prices positive: True


In [10]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved /Users/crablan/Desktop/bootcamp_haoting_lan/data/raw/api_source-yfinance_symbol-AAPL_20260817-194836.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [11]:
SCRAPE_URL = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
    "User-Agent": "Mozilla/5.0"
}

try:
    resp = requests.get(
        SCRAPE_URL,
        headers=headers,
        timeout=30
    )
    resp.raise_for_status()

    soup = BeautifulSoup(resp.text, "html.parser")

    # Find the S&P 500 constituents table
    table = soup.find("table", id="constituents")

    if table is None:
        raise ValueError("Could not find S&P 500 constituents table.")

    rows = []

    for tr in table.find_all("tr"):
        cells = [
            c.get_text(" ", strip=True)
            for c in tr.find_all(["th", "td"])
        ]

        if cells:
            rows.append(cells)

    header = rows[0]
    data = rows[1:]

    df_scrape = pd.DataFrame(data, columns=header)

    print(df_scrape.head())
    print("\nShape:", df_scrape.shape)

except Exception as e:
    print("Scrape failed:", e)

  Symbol             Security             GICS Sector  \
0    MMM                   3M             Industrials   
1    AOS          A. O. Smith             Industrials   
2    ABT  Abbott Laboratories             Health Care   
3   ABBV               AbbVie             Health Care   
4    ACN            Accenture  Information Technology   

                GICS Sub-Industry    Headquarters Location  Date added  \
0        Industrial Conglomerates    Saint Paul, Minnesota  1957-03-04   
1               Building Products    Milwaukee , Wisconsin  2017-07-26   
2           Health Care Equipment  North Chicago, Illinois  1957-03-04   
3                   Biotechnology  North Chicago, Illinois  2012-12-31   
4  IT Consulting & Other Services         Dublin , Ireland  2011-07-06   

          CIK      Founded  
0  0000066740         1902  
1  0000091142         1916  
2  0000001800         1888  
3  0001551152  2013 (1888)  
4  0001467373         1989  

Shape: (503, 8)


In [12]:
required_scrape_cols = [
    "Symbol",
    "Security",
    "GICS Sector"
]

v_scrape = validate(
    df_scrape,
    required_scrape_cols
)

print("Validation results:")
print(v_scrape)

print("\nNA counts:")
print(
    df_scrape[required_scrape_cols]
    .isna()
    .sum()
)

print("\nBasic rules:")
print("Rows greater than 0:", len(df_scrape) > 0)

print(
    "Symbols are text:",
    df_scrape["Symbol"]
    .map(lambda x: isinstance(x, str))
    .all()
)

print(
    "Security names are non-empty:",
    df_scrape["Security"]
    .str.strip()
    .ne("")
    .all()
)

Validation results:
{'missing': [], 'shape': (503, 8), 'na_total': 0}

NA counts:
Symbol         0
Security       0
GICS Sector    0
dtype: int64

Basic rules:
Rows greater than 0: True
Symbols are text: True
Security names are non-empty: True


In [14]:
_ = save_csv(
    df_scrape,
    prefix="scrape",
    site="wikipedia",
    table="sp500"
)

Saved /Users/crablan/Desktop/bootcamp_haoting_lan/data/raw/scrape_site-wikipedia_table-sp500_20260817-194946.csv


In [13]:
_ = save_csv(df_scrape, prefix='scrape', site='example', table='markets')

Saved /Users/crablan/Desktop/bootcamp_haoting_lan/data/raw/scrape_site-example_table-markets_20260817-194919.csv


## Documentation
- API Source: (URL/endpoint/params)
- Scrape Source: (URL/table description)
- Assumptions & risks: (rate limits, selector fragility, schema changes)
- Confirm `.env` is not committed.

## Documentation

### API Source
- Source: Yahoo Finance through the `yfinance` Python package
- Ticker: AAPL
- Period: 3 months
- Interval: 1 day
- Fields used: date and adjusted close
- Validation: required columns, shape, missing-value counts, valid dates, and positive prices

### Scrape Source
- Source: Wikipedia — List of S&P 500 companies
- URL: https://en.wikipedia.org/wiki/List_of_S%26P_500_companies
- Table: S&P 500 constituents
- Fields checked: Symbol, Security, and GICS Sector
- Validation: required columns, shape, missing-value counts, text type, and non-empty company names

### Assumptions & Risks
The API data depends on the availability and format of the data returned through yfinance. Changes in the source or package may affect the ingestion process. The scraping workflow also depends on the structure of the Wikipedia table, so changes to the table ID or column names may cause the scraper to fail. In addition, both datasets represent information available at the time of ingestion and may change in future runs.

### Secrets
The `.env` file is stored locally and excluded from Git through `.gitignore`. Only `.env.example` is committed to the repository.